# 03. Return Dynamics & Rolling Volatility Estimators
### Mathematical Definitions
1. **Simple Return**: $R_t = \frac{P_t}{P_{t-1}} - 1$
2. **Logarithmic Return**: $r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(1 + R_t)$
3. **Annualized Rolling Volatility**: $\sigma_{ann, w} = \sqrt{252} \times \text{std}(r_{t-w+1:t})$
4. **Parkinson (1980) Range Volatility**:
   $$\sigma_{Parkinson}^2 = \frac{252}{4 \ln 2} \frac{1}{N} \sum_{i=1}^N \ln\left(\frac{High_i}{Low_i}\right)^2$$
5. **Garman-Klass (1980) Microstructure Volatility**:
   $$\sigma_{GK}^2 = \frac{252}{N} \sum_{i=1}^N \left[ 0.5 \ln\left(\frac{H_i}{L_i}\right)^2 - (2\ln 2 - 1) \ln\left(\frac{C_i}{O_i}\right)^2 \right]$$


In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.volatility import (
    compute_returns,
    compute_rolling_volatility,
    compute_parkinson_volatility,
    compute_garman_klass_volatility
)

df = pd.read_csv("../data/processed/nifty50_daily_processed.csv", parse_dates=["Date"], index_col="Date")
df = compute_returns(df, price_col="Close")
df = compute_rolling_volatility(df, return_col="log_return", windows=[5, 10, 20, 30, 60])
df["vol_parkinson_20d"] = compute_parkinson_volatility(df, window=20)
df["vol_garman_klass_20d"] = compute_garman_klass_volatility(df, window=20)

df[["log_return", "vol_ann_20d", "vol_parkinson_20d", "vol_garman_klass_20d"]].tail()


In [ ]:
# Compare Volatility Estimators
plt.figure(figsize=(11, 5))
plt.plot(df.index, df["vol_ann_20d"] * 100, label="Close-to-Close Rolling (20d)", lw=1.2)
plt.plot(df.index, df["vol_parkinson_20d"] * 100, label="Parkinson Range (20d)", lw=1.2, alpha=0.8)
plt.plot(df.index, df["vol_garman_klass_20d"] * 100, label="Garman-Klass (20d)", lw=1.2, alpha=0.8)
plt.title("NIFTY 50: Comparison of Realized Volatility Estimators")
plt.ylabel("Annualized Volatility (%)")
plt.xlabel("Date")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()
